# uv 멀티스테이지 빌드

Docker Desktop의 실습 전용 이미지와 스택만 사용해 결과를 확인합니다.

## 실행 환경 준비

Docker 명령과 실습 경로를 준비합니다.

In [1]:
# Docker 명령을 일정한 폴더에서 실행하고 실패도 결과로 받습니다.
from pathlib import Path
import json, subprocess, time, urllib.request
# 챕터 폴더와 notebooks 폴더 어디에서 시작해도 제공 자료를 찾습니다.
candidates = [parent / "labs/part7/ch32-docker" for parent in (Path.cwd(), *Path.cwd().parents)]
candidates += [Path.cwd(), Path.cwd().parent]
LAB = next(path for path in candidates if (path / "Dockerfile.baseline").is_file())
def run(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(args, cwd=LAB, text=True, capture_output=True, check=check)
version = run("docker", "version", "--format", "{{.Server.Version}}")
print("Docker Engine", version.stdout.strip())
assert version.stdout.strip()

Docker Engine 29.4.1


## 기본 실행

최적화 Dockerfile을 빌드하고 uv가 최종 이미지에 없는지 확인합니다.

In [2]:
# builder에는 uv가 있지만 runtime에는 실행 환경만 복사합니다.
run("docker", "build", "-f", "Dockerfile", "-t", "ch32-optimized:local", ".")
python_version = run("docker", "run", "--rm", "ch32-optimized:local", "python", "--version")
uv_probe = run("docker", "run", "--rm", "ch32-optimized:local", "sh", "-c", "command -v uv", check=False)
print(python_version.stdout.strip(), "uv exit", uv_probe.returncode)
assert python_version.stdout.startswith("Python 3.12") and uv_probe.returncode != 0

Python 3.12.14 uv exit 127


## 실패 재현

존재하지 않는 stage를 지정하면 빌드가 시작되기 전에 실패합니다.

In [3]:
# stage 이름 오타를 실제 빌드 오류로 확인합니다.
bad = run("docker", "build", "--target", "buidler", "-f", "Dockerfile", ".", check=False)
message = bad.stderr + bad.stdout
print("exit", bad.returncode, "unknown target", "buidler" in message)
assert bad.returncode != 0 and "buidler" in message

exit 1 unknown target True


## 수정

정확한 builder stage를 지정하면 의존성 환경까지 빌드됩니다.

In [4]:
# 정확한 stage 이름으로 중간 산출물을 검사합니다.
run("docker", "build", "--target", "builder", "-t", "ch32-builder:local", ".")
probe = run("docker", "run", "--rm", "ch32-builder:local", "sh", "-c", "test -x /app/.venv/bin/uvicorn")
print("builder venv ready", probe.returncode == 0)
assert probe.returncode == 0

builder venv ready True
